In [ ]:
%%capture
%pip install langchain chromadb sentence-transformers beautifulsoup4 requests langchain langchain-community langchain-huggingface langchain-chroma faiss-cpu

In [61]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from transformers import AutoTokenizer

*Das HF_TOKEN muss mit `setx HF_TOKEN "TOKENWERT"` gesetzt werden*

In [62]:
def fetch_threads_list(category_url):
    resp = requests.get(category_url + ".json")
    data = resp.json()
    threads = []
    for t in data["topic_list"]["topics"]:
        threads.append({
            "topic_id": t["id"],
            "topic_slug": t["slug"],
            "title": t["title"]
        })
    return threads

def fetch_thread_posts(topic_id, topic_slug):
    url = f"https://forums.forza.net/t/{topic_slug}/{topic_id}.json"
    resp = requests.get(url)
    data = resp.json()
    posts = []
    for p in data["post_stream"]["posts"]:
        posts.append({
            "post_id": p["id"],
            "author_name": p["name"],
            "username": p["username"],
            "created_at": p["created_at"],
            "content_text": BeautifulSoup(p["cooked"], "html.parser").get_text(),
            "topic_id": p["topic_id"],
            "topic_slug": p["topic_slug"]
        })
    return posts

CATEGORY_URL = "https://forums.forza.net/tags/c/community-hub/ugc/tuning/148/fh5"
threads = fetch_threads_list(CATEGORY_URL)

all_posts = []
for t in threads:
    all_posts.extend(fetch_thread_posts(t["topic_id"], t["topic_slug"]))


In [63]:
%%capture
# Create documents for RAG
docs = []
for p in all_posts:
    docs.append(Document(page_content=p["content_text"], metadata={
        "author": p["author_name"],
        "topic": p["topic_slug"],
        "created_at": p["created_at"]
    }))

# Text Embedding & Chunking 
embedding_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", use_fast=False)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    embedding_tokenizer,
    chunk_size=500,
    chunk_overlap=50
)
chunked_docs = text_splitter.split_documents(docs)

# --- Embeddings & Vektorstore aufbauen --- #
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   encode_kwargs={"normalize_embeddings": True})
vectorstore = FAISS.from_documents(chunked_docs, embedding=embeddings)

In [37]:
vectordb_path = "../data/vectordb"
vectorstore.save_local(vectordb_path)

In [60]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
question = "How do I setup my car best in Forza Horizon 5?"
retrieved_docs = retriever.invoke(question)
retrieved_docs[2].page_content

'Conclusion: Mastering the Art of Forza Tuning\nOptimizing car performance in Forza Horizon 5 through tuning is a multifaceted process that requires a thorough understanding of various parameters and their intricate effects on vehicle dynamics. By carefully adjusting tire pressure, gearing, alignment, suspension, aerodynamics, brakes, and the differential, players can significantly enhance their car’s handling, grip, acceleration, and overall competitiveness across a wide range of racing disciplines and track conditions. Utilizing the in-game telemetry provides invaluable data for making informed tuning decisions, allowing for precise adjustments based on real-time vehicle behavior. Engaging with the active Forza Horizon 5 community offers a wealth of knowledge through shared guides, discussions, and downloadable tuning setups, providing a valuable resource for both novice and experienced tuners. Ultimately, mastering the art of Forza tuning is an ongoing journey of experimentation and